In [1]:
import pandas as pd
import numpy as np

In [3]:
# 1. GENERATE SAMPLE E-COMMERCE SALES & CHURN DATASET

np.random.seed(42)
dates = pd.date_range(start="2023-01-01", end="2023-12-31", freq="D")
n_rows = 1000

data = {
    'OrderID': range(1000, 1000 + n_rows),
    'CustomerID': np.random.randint(100, 200, size=n_rows),
    'OrderDate': np.random.choice(dates, size=n_rows),
    'Category': np.random.choice(['Electronics', 'Clothing', 'Home & Kitchen', 'Books'], size=n_rows),
    'Quantity': np.random.randint(1, 5, size=n_rows),
    'UnitPrice': np.random.uniform(10, 200, size=n_rows),
    'CustomerSegment': np.random.choice(['VIP', 'Regular', 'New'], size=n_rows, p=[0.2, 0.5, 0.3])
}

df = pd.DataFrame(data)

In [4]:
# 2. FEATURE ENGINEERING & DATE EXTRACTION

df['OrderDate'] = pd.to_datetime(df['OrderDate'])

# Total Sales Calculation
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

# Date Feature Extractions
df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.strftime('%B')
df['MonthNo'] = df['OrderDate'].dt.month
df['Quarter'] = df['OrderDate'].dt.to_period('Q')

print("=== FIRST 5 ROWS WITH FEATURE ENGINEERING ===")
print(df.head())

=== FIRST 5 ROWS WITH FEATURE ENGINEERING ===
   OrderID  CustomerID  OrderDate        Category  Quantity   UnitPrice  \
0     1000         151 2023-02-03  Home & Kitchen         2  167.513369   
1     1001         192 2023-04-21        Clothing         4  198.340013   
2     1002         114 2023-09-21  Home & Kitchen         2  165.802400   
3     1003         171 2023-08-24  Home & Kitchen         4  142.027651   
4     1004         160 2023-03-24        Clothing         2  111.173613   

  CustomerSegment  TotalSales  Year      Month  MonthNo Quarter  
0         Regular  335.026738  2023   February        2  2023Q1  
1         Regular  793.360052  2023      April        4  2023Q2  
2             New  331.604799  2023  September        9  2023Q3  
3         Regular  568.110604  2023     August        8  2023Q3  
4         Regular  222.347226  2023      March        3  2023Q1  


In [5]:
# 3. PIVOT TABLES & TREND CALCULATIONS

print("\n=== MONTHLY REVENUE TREND ===")
monthly_sales = df.groupby(['MonthNo', 'Month'])['TotalSales'].sum().reset_index()
print(monthly_sales[['Month', 'TotalSales']])

print("\n=== PIVOT TABLE: SALES BY CATEGORY & CUSTOMER SEGMENT ===")
segment_pivot = pd.pivot_table(
    df,
    values='TotalSales',
    index='Category',
    columns='CustomerSegment',
    aggfunc='sum',
    margins=True
)
print(segment_pivot)

# Identifying Top Customer Segments by Total Spend
print("\n=== REVENUE & ORDER COUNT BY CUSTOMER SEGMENT ===")
segment_summary = df.groupby('CustomerSegment').agg(
    TotalRevenue=('TotalSales', 'sum'),
    AverageOrderValue=('TotalSales', 'mean'),
    TotalOrders=('OrderID', 'count')
).reset_index()
print(segment_summary)


=== MONTHLY REVENUE TREND ===
        Month    TotalSales
0     January  28768.412587
1    February  21868.868514
2       March  19380.330330
3       April  20823.223415
4         May  18394.373132
5        June  26630.772768
6        July  15613.198097
7      August  23643.519677
8   September  21321.278427
9     October  24261.723949
10   November  25607.582157
11   December  24184.987922

=== PIVOT TABLE: SALES BY CATEGORY & CUSTOMER SEGMENT ===
CustomerSegment           New        Regular           VIP            All
Category                                                                 
Books            12908.705573   42722.259399  14874.904844   70505.869817
Clothing         21055.276381   33860.440661  14309.536534   69225.253576
Electronics      25539.750036   30136.455751  15883.820270   71560.026057
Home & Kitchen   19651.119906   29574.349823   9981.651798   59207.121527
All              79154.851897  136293.505634  55049.913445  270498.270976

=== REVENUE & ORDER COUNT B